In [34]:
# ============================================================
# Cell 1 | 库加载 & 原始数据读入
# ============================================================
# 依赖：tidyverse（dplyr/readr/tidyr）
# 数据目录：./data/（相对于 notebook 所在目录）
# ------------------------------------------------------------

suppressPackageStartupMessages(library(tidyverse))

DATA_PATH <- "data/"

race         <- read_csv(paste0(DATA_PATH, "ALL_raw_race_results.csv"),   show_col_types = FALSE)
quali        <- read_csv(paste0(DATA_PATH, "ALL_raw_quali_laptimes.csv"), show_col_types = FALSE)
meta         <- read_csv(paste0(DATA_PATH, "ALL_raw_event_meta.csv"),     show_col_types = FALSE)
weather      <- read_csv(paste0(DATA_PATH, "ALL_raw_weather_summary.csv"),show_col_types = FALSE)
track_status <- read_csv(paste0(DATA_PATH, "ALL_raw_track_status.csv"),   show_col_types = FALSE)
overtakes    <- read_csv(paste0(DATA_PATH, "ALL_clean_overtakes_season.csv"), show_col_types = FALSE)

cat("Loaded rows — race:", nrow(race),
    "| quali:", nrow(quali),
    "| meta:", nrow(meta),
    "| weather:", nrow(weather),
    "| track_status:", nrow(track_status),
    "| overtakes:", nrow(overtakes), "\n")
cat("Years in race data:", paste(sort(unique(race$Year)), collapse = ", "), "\n")

Loaded rows — race: 3502 | quali: 3474 | meta: 195 | weather: 175 | track_status: 12036 | overtakes: 5981 
Years in race data: 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026 


In [35]:
# ============================================================
# Cell 2 | 因变量（DV）
# ------------------------------------------------------------
# 主 DV（Hallila 2024 复现）：
#   internal_overtakes：clean 超车中 IsTeammate == TRUE 的超车数
#   external_overtakes：clean 超车中 IsTeammate == FALSE 的超车数
# ============================================================

# ── 主 DV ─────────────────────────────────────────────────
dv_internal <- overtakes %>%
  filter(IsTeammate == TRUE) %>%
  group_by(Year, Round, DriverAbbr = Overtaker) %>%
  summarise(internal_overtakes = n(), .groups = "drop")

dv_external <- overtakes %>%
  filter(IsTeammate == FALSE) %>%
  group_by(Year, Round, DriverAbbr = Overtaker) %>%
  summarise(external_overtakes = n(), .groups = "drop")

# ── 汇报 ──────────────────────────────────────────────────
cat("DV counts:\n")
cat("  internal_overtakes records:", nrow(dv_internal),
    " mean:", round(mean(dv_internal$internal_overtakes), 2), "\n")
cat("  external_overtakes records:", nrow(dv_external),
    " mean:", round(mean(dv_external$external_overtakes), 2), "\n")

DV counts:
  internal_overtakes records: 399  mean: 1.09 
  external_overtakes records: 2080  mean: 2.67 


In [36]:
# ============================================================
# Cell 3 | 自变量（IV）— competitive_threats & competitive_opportunities
# ------------------------------------------------------------
# 逻辑（Hallila 2024 原文）：
#   赛前积分差 ≤ remaining_races × max_pts_per_race_per_team
#   的车队才被视为"有效竞争对手"
#
# 参数选择：
#   max_pts_per_race_per_team = 44（两名车手上限：25+18+1 最快圈）
#   时间基准：赛前（cum_pts_before = 上一站结束后的累计积分，第 1 站 = 0）
#   total_rounds 每年赛历长度（2026 硬编码为 24）
#
# Direction D 扩展（Season Phase + 竞争活跃度 + 数学淘汰）：
#   season_phase                 : Round / total_rounds，范围 (0, 1]
#   tournament_criticality       : (CT + CO) / (n_teams − 1)，0–1 连续指数
#   is_mathematically_locked_out : CO == 0 → 无法超越任何对手
#   is_mathematically_safe       : CT == 0 → 无队伍能超越本队
# ============================================================

MAX_PTS_PER_RACE <- 44L   # 每站每队理论最高积分

total_rounds_lookup <- c(
  "2018" = 21, "2019" = 21, "2020" = 17,
  "2021" = 22, "2022" = 22, "2023" = 23,
  "2024" = 24, "2025" = 24, "2026" = 24
)

# Step 1：每队每站积分（队内两名车手积分之和）
team_pts <- race %>%
  group_by(Year, Round, TeamName) %>%
  summarise(round_pts = sum(Points, na.rm = TRUE), .groups = "drop") %>%
  arrange(Year, TeamName, Round)

# Step 2：赛前累积积分 = lag(cumsum)；第 1 站赛前积分为 0
team_pts <- team_pts %>%
  group_by(Year, TeamName) %>%
  mutate(
    cum_pts_after  = cumsum(round_pts),
    cum_pts_before = lag(cum_pts_after, default = 0L)
  ) %>%
  ungroup()

# Step 3：附加赛历参数
team_pts <- team_pts %>%
  mutate(
    total_rounds    = total_rounds_lookup[as.character(Year)],
    remaining_races = total_rounds - Round,          # 本站之后还剩几站（含本站后）
    max_obtainable  = remaining_races * MAX_PTS_PER_RACE
  )

# Step 4：自连接 —— 计算每队面对所有其他车队的威胁数/机会数
threats_opps <- team_pts %>%
  inner_join(team_pts,
             by     = c("Year", "Round"),
             suffix  = c(".focal", ".other")) %>%
  filter(TeamName.focal != TeamName.other) %>%
  group_by(Year, Round, TeamName = TeamName.focal) %>%
  summarise(
    # 威胁：本队积分 > 对手积分，但差距 ≤ 对手在剩余赛站可获得的最高分
    competitive_threats = sum(
      (cum_pts_before.focal > cum_pts_before.other) &
      ((cum_pts_before.focal - cum_pts_before.other) <= max_obtainable.focal),
      na.rm = TRUE
    ),
    # 机会：本队积分 < 对手积分，但差距 ≤ 本队在剩余赛站可获得的最高分
    competitive_opportunities = sum(
      (cum_pts_before.focal < cum_pts_before.other) &
      ((cum_pts_before.other - cum_pts_before.focal) <= max_obtainable.focal),
      na.rm = TRUE
    ),
    .groups = "drop"
  )

# ── Direction D：赛季时段 + 竞争活跃度指数 + 数学淘汰标记 ──────
# 每年各站参赛队数（动态，兼容赛历变化）
n_teams_yr <- race %>%
  group_by(Year) %>%
  summarise(n_teams = n_distinct(TeamName), .groups = "drop")

threats_opps <- threats_opps %>%
  # 附加 total_rounds（用于计算 season_phase）
  left_join(
    team_pts %>% distinct(Year, Round, TeamName, total_rounds),
    by = c("Year", "Round", "TeamName")
  ) %>%
  left_join(n_teams_yr, by = "Year") %>%
  mutate(
    # 赛季进度：本站在全年赛历中的位置
    season_phase = Round / total_rounds,
    # 竞争活跃度：CT+CO 占理论最大值（n_teams−1）的比例
    tournament_criticality = (competitive_threats + competitive_opportunities) /
                               (n_teams - 1),
    # 数学淘汰二元标记
    is_mathematically_locked_out = as.integer(competitive_opportunities == 0),
    is_mathematically_safe       = as.integer(competitive_threats == 0)
  ) %>%
  select(-total_rounds, -n_teams)

cat("IV rows:", nrow(threats_opps), "\n")
cat("Threats    — mean:", round(mean(threats_opps$competitive_threats), 2),
    " range:", min(threats_opps$competitive_threats), "–", max(threats_opps$competitive_threats), "\n")
cat("Opps       — mean:", round(mean(threats_opps$competitive_opportunities), 2),
    " range:", min(threats_opps$competitive_opportunities), "–", max(threats_opps$competitive_opportunities), "\n")
cat("Season phase     — mean:", round(mean(threats_opps$season_phase), 2),
    " range: [", round(min(threats_opps$season_phase), 2), ",",
    round(max(threats_opps$season_phase), 2), "]\n")
cat("Criticality      — mean:", round(mean(threats_opps$tournament_criticality), 2), "\n")
cat("Locked out freq :", round(mean(threats_opps$is_mathematically_locked_out), 3),
    " | Safe freq:", round(mean(threats_opps$is_mathematically_safe), 3), "\n")

Warning message in inner_join(., team_pts, by = c("Year", "Round"), suffix = c(".focal", :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


IV rows: 1752 
Threats    — mean: 3.31  range: 0 – 10 
Opps       — mean: 3.31  range: 0 – 9 
Season phase     — mean: 0.51  range: [ 0.04 , 1 ]
Criticality      — mean: 0.73 
Locked out freq : 0.225  | Safe freq: 0.253 


In [37]:
# ============================================================
# Cell 3b | 扩展变量：Direction A（队内地位）+ Direction B（车手级 CT/CO）
# ============================================================
#
# Direction A — 队内地位调节（Within-Team Rank）
#   理论：地位竞争理论 / 社会比较。落后队友的车手承受队内压力，
#         当面临外部威胁时更可能采取激进竞争行动。
#   变量：
#     within_team_pts_gap_before  : focal 赛前累积分 − 队友赛前累积分
#                                   正值 = 本队领先，负值 = 落后，第 1 站 = 0
#     is_leading_teammate         : 1 if gap > 0，else 0
#
# Direction B — 双重锦标赛冲突（Driver-Level CT / CO）
#   理论：多委托代理。车手个人锦标赛目标 vs 车队锦标赛目标冲突时，
#         个人淘汰压力会稀释/放大对车队竞争行动的响应。
#   常数：MAX_PTS_DRIVER = 26（单名车手单站理论最高：25 + 1 最快圈）
#   变量：
#     driver_ct              : 本车手赛前积分高于 other，但 other 理论上仍可追上
#     driver_co              : 本车手赛前积分低于 other，但本车手理论上仍可追上
#     driver_pts_gap_to_p1   : 积分领袖赛前积分 − focal 赛前积分（≥ 0）
# ============================================================

MAX_PTS_DRIVER <- 26L  # 单车手单站最高分（25 分位 + 1 最快圈）

# ── Step 1：车手级赛前累积积分（与 team_pts 逻辑一致）──────────
driver_pts <- race %>%
  select(Year, Round, DriverAbbr, TeamName, Points) %>%
  arrange(Year, DriverAbbr, Round) %>%
  group_by(Year, DriverAbbr) %>%
  mutate(
    driver_cum_pts_after  = cumsum(coalesce(Points, 0L)),
    driver_cum_pts_before = lag(driver_cum_pts_after, default = 0L)
  ) %>%
  ungroup() %>%
  select(Year, Round, DriverAbbr, TeamName, driver_cum_pts_before)

# ── Direction A：队内地位 ────────────────────────────────────
within_team_rank <- driver_pts %>%
  inner_join(
    driver_pts %>% select(Year, Round, TeamName,
                          DriverAbbr_other = DriverAbbr,
                          other_driver_pts = driver_cum_pts_before),
    by = c("Year", "Round", "TeamName")
  ) %>%
  filter(DriverAbbr != DriverAbbr_other) %>%
  group_by(Year, Round, DriverAbbr) %>%
  summarise(
    within_team_pts_gap_before = first(driver_cum_pts_before) -
                                  first(other_driver_pts),
    is_leading_teammate = as.integer(first(driver_cum_pts_before) >
                                      first(other_driver_pts)),
    .groups = "drop"
  )

# ── Direction B：车手级 CT / CO ──────────────────────────────
driver_pts_ext <- driver_pts %>%
  mutate(
    total_rounds          = total_rounds_lookup[as.character(Year)],
    remaining_races       = total_rounds - Round,
    max_obtainable_driver = remaining_races * MAX_PTS_DRIVER
  )

# 每站赛季积分领袖积分（赛前，用于计算 gap to P1）
driver_p1_pts <- driver_pts_ext %>%
  group_by(Year, Round) %>%
  summarise(p1_pts = max(driver_cum_pts_before, na.rm = TRUE), .groups = "drop")

# 全场车手自连接
driver_level_ct_co <- driver_pts_ext %>%
  inner_join(
    driver_pts_ext %>% select(Year, Round,
                               DriverAbbr_other  = DriverAbbr,
                               other_pts_before  = driver_cum_pts_before,
                               other_max_obt     = max_obtainable_driver),
    by = c("Year", "Round")
  ) %>%
  filter(DriverAbbr != DriverAbbr_other) %>%
  group_by(Year, Round, DriverAbbr) %>%
  summarise(
    driver_ct = sum(
      (driver_cum_pts_before > other_pts_before) &
      ((driver_cum_pts_before - other_pts_before) <= other_max_obt),
      na.rm = TRUE
    ),
    driver_co = sum(
      (driver_cum_pts_before < other_pts_before) &
      ((other_pts_before - driver_cum_pts_before) <= max_obtainable_driver),
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  left_join(driver_p1_pts, by = c("Year", "Round")) %>%
  left_join(
    driver_pts %>% select(Year, Round, DriverAbbr, driver_cum_pts_before),
    by = c("Year", "Round", "DriverAbbr")
  ) %>%
  mutate(driver_pts_gap_to_p1 = p1_pts - driver_cum_pts_before) %>%
  select(-p1_pts, -driver_cum_pts_before)

# ── 汇报 ─────────────────────────────────────────────────────
cat("=== Direction A — Within-Team Rank ===\n")
cat("  Rows:", nrow(within_team_rank), "\n")
cat("  Leading teammate %:", round(mean(within_team_rank$is_leading_teammate) * 100, 1), "%\n")
cat("  Pts gap — mean:", round(mean(within_team_rank$within_team_pts_gap_before), 1),
    " sd:", round(sd(within_team_rank$within_team_pts_gap_before), 1), "\n\n")

cat("=== Direction B — Driver-Level CT / CO ===\n")
cat("  Rows:", nrow(driver_level_ct_co), "\n")
cat("  driver_ct — mean:", round(mean(driver_level_ct_co$driver_ct), 2),
    " range:", min(driver_level_ct_co$driver_ct), "–", max(driver_level_ct_co$driver_ct), "\n")
cat("  driver_co — mean:", round(mean(driver_level_ct_co$driver_co), 2),
    " range:", min(driver_level_ct_co$driver_co), "–", max(driver_level_ct_co$driver_co), "\n")
cat("  pts_gap_to_p1 — mean:", round(mean(driver_level_ct_co$driver_pts_gap_to_p1), 1), "\n")

Warning message in inner_join(., driver_pts %>% select(Year, Round, TeamName, DriverAbbr_other = DriverAbbr, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 22 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message in inner_join(., driver_pts_ext %>% select(Year, Round, DriverAbbr_other = DriverAbbr, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


=== Direction A — Within-Team Rank ===
  Rows: 3500 
  Leading teammate %: 42.9 %
  Pts gap — mean: 0  sd: 47.6 

=== Direction B — Driver-Level CT / CO ===
  Rows: 3502 
  driver_ct — mean: 6.97  range: 0 – 21 
  driver_co — mean: 6.97  range: 0 – 19 
  pts_gap_to_p1 — mean: 147.6 


In [38]:
# ============================================================
# Cell 4 | 调节变量（Moderator）— quali_gap_seconds（资源禀赋代理）
# ------------------------------------------------------------
# 定义（对齐 Hallila 2024）：
#   quali_gap_seconds = 该队最快排位圈时 − 全场杆位圈时
#   车队层级：取队内两名车手中较快者（min(BestLapTime_s) per team）
# ============================================================

# Step 1：每站杆位时间（全场最快排位圈）
pole_times <- quali %>%
  group_by(Year, Round) %>%
  summarise(pole_time = min(BestLapTime_s, na.rm = TRUE), .groups = "drop")

# Step 2：每名车手相对杆位的差距
driver_gaps <- quali %>%
  left_join(pole_times, by = c("Year", "Round")) %>%
  mutate(driver_gap = BestLapTime_s - pole_time)

# Step 3：车队层级 —— 取队内最小 gap（最具竞争力的车手代表车队）
team_quali_gap <- driver_gaps %>%
  group_by(Year, Round, TeamName = Team) %>%
  summarise(quali_gap_seconds = min(driver_gap, na.rm = TRUE), .groups = "drop")

cat("Quali gap rows:", nrow(team_quali_gap), "\n")
cat("Poll sitter gap = 0 check:",
    sum(team_quali_gap$quali_gap_seconds == 0),
    "records with gap = 0 (should equal total races)\n")
cat("Range:", round(min(team_quali_gap$quali_gap_seconds, na.rm=TRUE), 3),
    "–", round(max(team_quali_gap$quali_gap_seconds, na.rm=TRUE), 3), "s\n")

Quali gap rows: 1752 
Poll sitter gap = 0 check: 175 records with gap = 0 (should equal total races)
Range: 0 – 22.252 s


In [39]:
# ============================================================
# Cell 5 | 控制变量（Controls）
# ============================================================
# 5a  driver_experience  ：该场之前车手累计参赛场次（0 = 生涯首战）
# 5b  sc_lap_fraction    ：安全车（SC）圈占比（TrackStatus 含 "4"）
#     vsc_lap_fraction   ：虚拟安全车（VSC）圈占比（TrackStatus 含 "5"）
# 5c  赛道控制           ：CircuitType_street, DRS_Zones, IsSprintWeekend
# 5d  天气（已在 Cell 1 直接加载，Cell 6 直接 join，略）
# ============================================================

# ── 5a：车手经验（已参赛场次，不含本场） ──────────────────────
# 按时间正序排列，row_number()-1 即为之前参赛场次
driver_experience <- race %>%
  arrange(Year, Round) %>%
  group_by(DriverAbbr) %>%
  mutate(driver_experience = row_number() - 1L) %>%
  ungroup() %>%
  select(Year, Round, DriverAbbr, driver_experience)

# ── 5b：赛道状态 — SC/VSC 圈数比例 ───────────────────────────
# track_status 每行为单支车手的一圈（多车手同圈会有重复）
# 先去重到 (Year, Round, LapNumber) 层级，再统计状态比例
sc_vsc <- track_status %>%
  filter(!is.na(TrackStatus), TrackStatus != "") %>%
  distinct(Year, Round, LapNumber, TrackStatus) %>%
  group_by(Year, Round) %>%
  summarise(
    sc_lap_fraction  = mean(str_detect(TrackStatus, "4"), na.rm = TRUE),
    vsc_lap_fraction = mean(str_detect(TrackStatus, "5"), na.rm = TRUE),
    .groups = "drop"
  )

# ── 5c：赛事控制变量（赛道类型 / DRS / Sprint） ────────────────
event_ctrl <- meta %>%
  mutate(
    CircuitType_street = as.integer(CircuitType == "street"),
    IsSprintWeekend    = as.integer(EventFormat  == "sprint")
  ) %>%
  select(Year, Round, CircuitType_street, CircuitLength_km, DRS_Zones, IsSprintWeekend)

cat("driver_experience rows:", nrow(driver_experience),
    " | max experience:", max(driver_experience$driver_experience), "\n")
cat("sc_vsc rows:", nrow(sc_vsc),
    " | races with any SC:", sum(sc_vsc$sc_lap_fraction > 0),
    " | races with any VSC:", sum(sc_vsc$vsc_lap_fraction > 0), "\n")
cat("event_ctrl rows:", nrow(event_ctrl),
    " | Sprint weeks:", sum(event_ctrl$IsSprintWeekend), "\n")

driver_experience rows: 3502  | max experience: 174 
sc_vsc rows: 175  | races with any SC: 99  | races with any VSC: 19 
event_ctrl rows: 195  | Sprint weeks: 6 


In [40]:
# ============================================================
# Cell 6 | 最终合并 & 输出
# ------------------------------------------------------------
# 基础层：race_results（driver × round）
# 依次 left_join：DV → IV/调节 → 扩展变量 → 控制变量
# DV 计数缺失（该场无超车）填 0
# 保留条件：GridPosition 不为 NA（过滤 DNS/未实际参赛车手）
#
# 新增列（扩展方向 A、B、D，E 在 Cell 7 单独生成 panel_lap）：
#   Direction D : season_phase, tournament_criticality,
#                 is_mathematically_locked_out, is_mathematically_safe
#                 （已合并进 threats_opps，自动随该表 join）
#   Direction A : within_team_pts_gap_before, is_leading_teammate
#   Direction B : driver_ct, driver_co, driver_pts_gap_to_p1
#   Direction E : times_overtaken_total（被超车次数，当站控制变量）
# ============================================================

# ── 预计算 Direction E 控制变量（被超车总次数，按站汇总）──────
times_overtaken <- overtakes %>%
  group_by(Year, Round, DriverAbbr = Overtaken) %>%
  summarise(times_overtaken_total = n(), .groups = "drop")

# ── 主合并链 ─────────────────────────────────────────────────
panel <- race %>%
  select(Year, Round, EventName,
         DriverAbbr, DriverNumber, TeamName,
         GridPosition, Position, ClassifiedLaps, TotalRaceLaps,
         Status, Points) %>%

  # ── DV ───────────────────────────────────────────────────
  left_join(dv_internal, by = c("Year", "Round", "DriverAbbr")) %>%
  left_join(dv_external, by = c("Year", "Round", "DriverAbbr")) %>%
  mutate(
    internal_overtakes = replace_na(internal_overtakes, 0L),
    external_overtakes = replace_na(external_overtakes, 0L)
  ) %>%

  # ── IV + Direction D（车队层级，Direction D 已合并进 threats_opps）
  left_join(threats_opps,   by = c("Year", "Round", "TeamName")) %>%

  # ── 调节变量（车队层级）────────────────────────────────────
  left_join(team_quali_gap, by = c("Year", "Round", "TeamName")) %>%

  # ── Direction A（队内地位，车手层级）──────────────────────
  left_join(within_team_rank, by = c("Year", "Round", "DriverAbbr")) %>%

  # ── Direction B（车手锦标赛 CT/CO，车手层级）──────────────
  left_join(driver_level_ct_co, by = c("Year", "Round", "DriverAbbr")) %>%

  # ── Direction E 控制：被超车次数 ────────────────────────────
  left_join(times_overtaken, by = c("Year", "Round", "DriverAbbr")) %>%
  mutate(times_overtaken_total = replace_na(times_overtaken_total, 0L)) %>%

  # ── 其他控制变量 ─────────────────────────────────────────────
  left_join(driver_experience, by = c("Year", "Round", "DriverAbbr")) %>%
  left_join(sc_vsc,            by = c("Year", "Round")) %>%
  left_join(event_ctrl,        by = c("Year", "Round")) %>%
  left_join(weather %>% select(-EventName), by = c("Year", "Round")) %>%

  # ── 仅保留有效参赛行 ─────────────────────────────────────────
  filter(!is.na(GridPosition))

# ── 汇报数据质量 ──────────────────────────────────────────────
cat("=== Final Panel ===\n")
cat("Dimensions:", nrow(panel), "rows ×", ncol(panel), "cols\n")
cat("Years:", paste(sort(unique(panel$Year)), collapse = ", "), "\n")
cat("Total race-rounds:", n_distinct(panel %>% select(Year, Round)), "\n")

cat("\n── DV Summary ──\n")
print(summary(panel %>% select(internal_overtakes, external_overtakes)))

cat("\n── IV / Direction D NA Check ──\n")
cat("  competitive_threats         missing:", sum(is.na(panel$competitive_threats)), "\n")
cat("  competitive_opportunities   missing:", sum(is.na(panel$competitive_opportunities)), "\n")
cat("  season_phase                missing:", sum(is.na(panel$season_phase)), "\n")
cat("  tournament_criticality      missing:", sum(is.na(panel$tournament_criticality)), "\n")
cat("  is_locked_out               missing:", sum(is.na(panel$is_mathematically_locked_out)), "\n")
cat("  is_safe                     missing:", sum(is.na(panel$is_mathematically_safe)), "\n")

cat("\n── Direction A NA Check ──\n")
cat("  within_team_pts_gap_before  missing:", sum(is.na(panel$within_team_pts_gap_before)), "\n")
cat("  is_leading_teammate         missing:", sum(is.na(panel$is_leading_teammate)), "\n")

cat("\n── Direction B NA Check ──\n")
cat("  driver_ct                   missing:", sum(is.na(panel$driver_ct)), "\n")
cat("  driver_co                   missing:", sum(is.na(panel$driver_co)), "\n")
cat("  driver_pts_gap_to_p1        missing:", sum(is.na(panel$driver_pts_gap_to_p1)), "\n")

cat("\n── Moderator & Controls NA Check ──\n")
cat("  quali_gap_seconds           missing:", sum(is.na(panel$quali_gap_seconds)), "\n")
cat("  times_overtaken_total       missing:", sum(is.na(panel$times_overtaken_total)), "\n")

# TeamName 命名一致性警告
n_no_iv <- sum(is.na(panel$competitive_threats))
if (n_no_iv > 0) {
  cat("\n⚠️  Rows missing competitive_threats — check TeamName consistency:\n")
  missing_teams <- panel %>%
    filter(is.na(competitive_threats)) %>%
    distinct(Year, TeamName) %>%
    arrange(Year, TeamName)
  print(missing_teams, n = 30)
}

# ── 保存 ──────────────────────────────────────────────────────
write_csv(panel, paste0(DATA_PATH, "panel_analysis.csv"))
saveRDS(panel,   paste0(DATA_PATH, "panel_analysis.rds"))
cat("\n✓ Saved:", paste0(DATA_PATH, "panel_analysis.csv"),
    "&", paste0(DATA_PATH, "panel_analysis.rds"), "\n")
cat("  Columns:", paste(names(panel), collapse = ", "), "\n")

=== Final Panel ===
Dimensions: 3499 rows × 40 cols
Years: 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026 
Total race-rounds: 175 

── DV Summary ──
 internal_overtakes external_overtakes
 Min.   :0.0000     Min.   : 0.000    
 1st Qu.:0.0000     1st Qu.: 0.000    
 Median :0.0000     Median : 1.000    
 Mean   :0.1237     Mean   : 1.586    
 3rd Qu.:0.0000     3rd Qu.: 2.000    
 Max.   :3.0000     Max.   :28.000    

── IV / Direction D NA Check ──
  competitive_threats         missing: 0 
  competitive_opportunities   missing: 0 
  season_phase                missing: 0 
  tournament_criticality      missing: 0 
  is_locked_out               missing: 0 
  is_safe                     missing: 0 

── Direction A NA Check ──
  within_team_pts_gap_before  missing: 2 
  is_leading_teammate         missing: 2 

── Direction B NA Check ──
  driver_ct                   missing: 0 
  driver_co                   missing: 0 
  driver_pts_gap_to_p1        missing: 0 

── Moderator & Contr

In [41]:
# ============================================================
# Cell 7 | Direction E — 圈级面板 panel_lap
# ------------------------------------------------------------
# 理论：竞争行动-反应序列。被超车触发 K 圈内的反击行为；
#        车队 CT 调节这种反击的强度。
#
# 数据粒度：driver × round × lap（race lap）
# 关键变量：
#   overtakes_this_lap      : 本圈 focal 车手的外部超车数
#   was_overtaken_this_lap  : 本圈 focal 车手被外部超车（指示变量）
#   times_overtaken_prev5   : 前 5 圈被超车累积次数（roll window）
#   times_overtaken_prev10  : 前 10 圈被超车累积次数（roll window）
#
# 从 panel 广播的变量（car-round level）：
#   competitive_threats, competitive_opportunities,
#   season_phase, tournament_criticality,
#   is_mathematically_locked_out, is_mathematically_safe,
#   driver_ct, driver_co, within_team_pts_gap_before
#
# 输出：data/panel_lap.csv + data/panel_lap.rds
# ============================================================

# ── Step 1：确定每站有效圈数范围 ─────────────────────────────
lap_range <- race %>%
  group_by(Year, Round) %>%
  summarise(max_lap = max(TotalRaceLaps, na.rm = TRUE), .groups = "drop")

# ── Step 2：构建完整 driver × round × lap 框架 ───────────────
# 每名参赛车手 × 该站所有圈次（含 DriverNumber）
driver_round_ids <- panel %>%
  select(Year, Round, DriverAbbr, DriverNumber, TeamName) %>%
  distinct()

panel_lap_base <- driver_round_ids %>%
  left_join(lap_range, by = c("Year", "Round")) %>%
  rowwise() %>%
  mutate(lap = list(seq_len(max_lap))) %>%
  unnest(lap) %>%
  select(-max_lap)

# ── Step 3：本圈超车 & 被超车（仅外部）─────────────────────
# 注意：超车数据列名为 LapNumber（非 Lap）
overtakes_per_lap <- overtakes %>%
  filter(IsTeammate == FALSE) %>%
  group_by(Year, Round, lap = LapNumber, DriverAbbr = Overtaker) %>%
  summarise(overtakes_this_lap = n(), .groups = "drop")

overtaken_per_lap <- overtakes %>%
  filter(IsTeammate == FALSE) %>%
  group_by(Year, Round, lap = LapNumber, DriverAbbr = Overtaken) %>%
  summarise(overtaken_this_lap = n(), .groups = "drop")

panel_lap <- panel_lap_base %>%
  left_join(overtakes_per_lap, by = c("Year", "Round", "DriverAbbr", "lap")) %>%
  left_join(overtaken_per_lap, by = c("Year", "Round", "DriverAbbr", "lap")) %>%
  mutate(
    overtakes_this_lap     = replace_na(overtakes_this_lap, 0L),
    overtaken_this_lap     = replace_na(overtaken_this_lap, 0L),
    was_overtaken_this_lap = as.integer(overtaken_this_lap > 0)
  )

# ── Step 4：滚动窗口 — 前 K 圈被超车次数 ─────────────────────
# 按 driver × round 内时间序列计算；不依赖外部包（纯 base R）
panel_lap <- panel_lap %>%
  arrange(Year, Round, DriverAbbr, lap) %>%
  group_by(Year, Round, DriverAbbr) %>%
  mutate(
    # cumsum 方法：当前圈之前（不含本圈）K 圈的被超车次数
    cum_overtaken = cumsum(overtaken_this_lap),
    times_overtaken_prev5  = pmax(0L,
      cum_overtaken - lag(cum_overtaken, n = 5,  default = 0L)),
    times_overtaken_prev10 = pmax(0L,
      cum_overtaken - lag(cum_overtaken, n = 10, default = 0L))
  ) %>%
  # 将 lag-based 前 K 圈改为"本圈之前"：shift by 1
  mutate(
    times_overtaken_prev5  = lag(times_overtaken_prev5,  default = 0L),
    times_overtaken_prev10 = lag(times_overtaken_prev10, default = 0L)
  ) %>%
  select(-cum_overtaken) %>%
  ungroup()

# ── Step 5：从 panel 广播赛站级别变量 ────────────────────────
panel_level_vars <- panel %>%
  select(Year, Round, DriverAbbr,
         competitive_threats, competitive_opportunities,
         season_phase, tournament_criticality,
         is_mathematically_locked_out, is_mathematically_safe,
         driver_ct, driver_co,
         within_team_pts_gap_before, is_leading_teammate,
         GridPosition, TotalRaceLaps)

panel_lap <- panel_lap %>%
  left_join(panel_level_vars, by = c("Year", "Round", "DriverAbbr"))

# ── 汇报 ─────────────────────────────────────────────────────
cat("=== panel_lap ===\n")
cat("Dimensions:", nrow(panel_lap), "rows ×", ncol(panel_lap), "cols\n")
cat("Years:", paste(sort(unique(panel_lap$Year)), collapse = ", "), "\n")
cat("Unique driver × round:", n_distinct(panel_lap %>% select(Year, Round, DriverAbbr)), "\n")
cat("\n── overtakes_this_lap distribution:\n")
print(table(panel_lap$overtakes_this_lap))
cat("── was_overtaken_this_lap:\n")
print(table(panel_lap$was_overtaken_this_lap))
cat("── times_overtaken_prev5 mean:", round(mean(panel_lap$times_overtaken_prev5, na.rm=TRUE), 3), "\n")
cat("── times_overtaken_prev10 mean:", round(mean(panel_lap$times_overtaken_prev10, na.rm=TRUE), 3), "\n")

# ── 保存 ─────────────────────────────────────────────────────
write_csv(panel_lap, paste0(DATA_PATH, "panel_lap.csv"))
saveRDS(panel_lap,   paste0(DATA_PATH, "panel_lap.rds"))
cat("\n✓ Saved:", paste0(DATA_PATH, "panel_lap.csv"),
    "&", paste0(DATA_PATH, "panel_lap.rds"), "\n")

=== panel_lap ===
Dimensions: 211174 rows × 23 cols
Years: 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026 
Unique driver × round: 3499 

── overtakes_this_lap distribution:

     0      1      2      3      4      5      6      7      8     10     12 
206003   4906    216     24     11      5      4      1      2      1      1 
── was_overtaken_this_lap:

     0      1 
206386   4788 
── times_overtaken_prev5 mean: 0.127 
── times_overtaken_prev10 mean: 0.244 

✓ Saved: data/panel_lap.csv & data/panel_lap.rds 
